# Contract Type Prediction using Machine Learning

This notebook demonstrates a machine learning workflow for automatically classifying financing agreements based on extracted contract information.

The objective is to predict the contract category from textual contract features.

In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv(
    "../data/processed/contracts.csv"
)

df.head()

,contract_type,borrower,lender,amount,clauses,jurisdiction,obligations,file
0,HEALTH FINANCING AGREEMENT,Julia Miller,FINANCIAL BANK OF AMERICA Inc.,"$42,751.00.","['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', '...","New York, NY",['CLAUSE THREE – PAYMENT: The BORROWER shall p...,Contract_16.pdf
1,MOTORCYCLE FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$36,160.00.","['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', '...","New York, NY",['CLAUSE THREE – PAYMENT: The BORROWER shall p...,Contract_4.pdf
2,HEALTH FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$40,267.00.","['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', '...","New York, NY",['CLAUSE THREE – PAYMENT: The BORROWER shall p...,Contract_5.pdf
3,STUDENT LOAN AGREEMENT,Richard Taylor,FINANCIAL BANK OF AMERICA Inc.,"$56,424.00.","['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', '...","New York, NY",['CLAUSE THREE – PAYMENT: The BORROWER shall p...,Contract_17.pdf
4,HEALTH FINANCING AGREEMENT,Mary Johnson,FINANCIAL BANK OF AMERICA Inc.,"$48,720.00.","['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', '...","New York, NY",['CLAUSE THREE – PAYMENT: The BORROWER shall p...,Contract_7.pdf


In [3]:
# Clean financing amount

df["amount_clean"] = (
    df["amount"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(".00.", "", regex=False)
    .astype(float)
)

df[["amount", "amount_clean"]].head()

,amount,amount_clean
0,"$42,751.00.",42751.0
1,"$36,160.00.",36160.0
2,"$40,267.00.",40267.0
3,"$56,424.00.",56424.0
4,"$48,720.00.",48720.0


In [4]:
df["amount_bucket"] = pd.cut(
    df["amount_clean"],
    bins=[0, 100000, 500000, 2000000],
    labels=["Small", "Medium", "Large"]
)

df[["amount_clean", "amount_bucket"]].head()

,amount_clean,amount_bucket
0,42751.0,Small
1,36160.0,Small
2,40267.0,Small
3,56424.0,Small
4,48720.0,Small


## Feature Engineering

Machine learning models require numerical representations of text.

TF-IDF (Term Frequency–Inverse Document Frequency) converts textual contract information into numerical feature vectors.

In [5]:
df["combined_text"] = (
    df["amount_bucket"].astype(str)
    + " "
    + df["clauses"].astype(str)
    + " "
    + df["obligations"].astype(str)
)

df[["combined_text", "contract_type"]].head()

,combined_text,contract_type
0,"Small ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTE...",HEALTH FINANCING AGREEMENT
1,"Small ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTE...",MOTORCYCLE FINANCING AGREEMENT
2,"Small ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTE...",HEALTH FINANCING AGREEMENT
3,"Small ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTE...",STUDENT LOAN AGREEMENT
4,"Small ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTE...",HEALTH FINANCING AGREEMENT


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(
    df["combined_text"]
)

y = df["contract_type"]

## Train-Test Split

The dataset is divided into training and testing subsets to evaluate model performance on unseen contracts.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
# Train Model
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Model Evaluation

The trained model is evaluated on the test dataset.
    

In [9]:
from sklearn.metrics import classification_report

predictions = model.predict(X_test)

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

                                 precision    recall  f1-score   support

        CAR FINANCING AGREEMENT       0.00      0.00      0.00         4
     HEALTH FINANCING AGREEMENT       0.17      1.00      0.29         1
REAL ESTATE FINANCING AGREEMENT       0.00      0.00      0.00         1

                       accuracy                           0.17         6
                      macro avg       0.06      0.33      0.10         6
                   weighted avg       0.03      0.17      0.05         6



In [13]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X,
    y,
    cv=5
)

print("Cross Validation Scores:", scores)
print("Average Accuracy:", scores.mean())

Cross Validation Scores: [0.33333333 0.33333333 0.33333333 0.33333333 0.33333333]
Average Accuracy: 0.3333333333333333


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [10]:
sample_contract = """
PAYMENT INSURANCE DEFAULT
monthly installments required
mandatory insurance
"""

sample_vector = vectorizer.transform(
    [sample_contract]
)

prediction = model.predict(
    sample_vector
)

print(
    "Predicted Contract Type:",
    prediction[0]
)

Predicted Contract Type: HEALTH FINANCING AGREEMENT


## Discussion

The classification model achieved limited predictive performance due to the small dataset size and the highly standardized nature of the synthetic contracts.

Most agreements contain similar clauses and obligations, resulting in limited discriminative information for machine learning.

Despite the low accuracy, the notebook successfully demonstrates a complete machine learning workflow including:

- Feature Engineering
- Text Vectorization (TF-IDF)
- Model Training
- Model Evaluation
- Automated Contract Classification

The workflow serves as a proof-of-concept for future large-scale legal document classification systems.

In [11]:
feature_names = vectorizer.get_feature_names_out()

print(
    "Number of Features:",
    len(feature_names)
)

feature_names[:30]

Number of Features: 25


array(['borrower', 'by', 'clause', 'contract', 'default', 'five', 'for',
       'guarantees', 'increased', 'installments', 'insurance',
       'jurisdiction', 'large', 'mandatory', 'monthly', 'must', 'pay',
       'payment', 'purpose', 'shall', 'small', 'term', 'termination',
       'the', 'three'], dtype=object)

In [12]:
import joblib

joblib.dump(
    model,
    "../data/processed/contract_classifier.pkl"
)

['../data/processed/contract_classifier.pkl']

In [15]:
print("Average Accuracy:", scores.mean())

Average Accuracy: 0.3333333333333333


## Results Discussion

The model achieved an average cross-validation accuracy of approximately 33%.

While this performance is modest, it is expected given the limited dataset size and the highly standardized structure of the synthetic contracts.

The objective of this notebook is to demonstrate the complete machine learning workflow for legal document classification, including:

- Text Feature Engineering
- TF-IDF Vectorization
- Model Training
- Cross-Validation
- Automated Contract Type Prediction

Future improvements would include larger datasets, richer textual features, and advanced NLP models.

# Conclusions

This notebook demonstrates a complete machine learning workflow for legal document classification.

Key stages included:

- Text preprocessing
- Feature engineering using TF-IDF
- Model training
- Performance evaluation
- Automated prediction

While the dataset is small, the workflow establishes a foundation for future large-scale contract classification systems.